# Multi-provider fidelity estimation benchmark on Amazon Braket

This notebook benchmarks a **unified estimated fidelity** against an **empirical success probability** measured on Amazon Braket hardware.

## What this notebook measures

For **gate-model QPUs** (IonQ, AQT, IQM, Rigetti), the notebook computes:

- `estimated_fidelity`: from provider calibration data mapped into a canonical model
- `actual_fidelity`: empirical probability mass on the **ideal support** of the benchmark circuit

This is intentionally a **success-probability style fidelity**, matching your model more closely than state fidelity from tomography would.

For **QuEra Aquila**, Braket exposes an **analog Hamiltonian simulation (AHS)** device rather than a gate-model circuit device. So this notebook uses a parallel success-probability benchmark for Aquila:
- `estimated_fidelity`: based on the neutral-atom terms available in your model (`P_fill`, `P_loss`, readout terms where available)
- `actual_fidelity`: fraction of shots landing in a chosen target solution family for the AHS benchmark

## Important caveats

1. `actual_fidelity` here means **empirical success probability**, not full quantum-state fidelity.
2. For **IonQ** and **AQT**, submission cells are present but commented out, per your request.
3. `device.properties` schemas can evolve over time; the helper accessors below are defensive and try multiple field names.
4. For QuEra, this notebook necessarily uses an **AHS benchmark**, not a gate-model circuit benchmark.

In [ ]:
!pip install amazon-braket-sdk networkx pandas matplotlib

import math
import time
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Iterable, Set, Any

import pandas as pd

from braket.aws import AwsDevice
from braket.circuits import Circuit
from braket.devices import LocalSimulator

try:
    from braket.ahs.atom_arrangement import AtomArrangement
    from braket.ahs.driving_field import DrivingField
    from braket.ahs.field import Field
    from braket.ahs.hamiltonian import Hamiltonian
    from braket.ahs.pattern import Pattern
    from braket.ahs.program import AnalogHamiltonianSimulation
    from braket.timings.time_series import TimeSeries
    AHS_AVAILABLE = True
except Exception as e:
    AHS_AVAILABLE = False
    print("AHS imports unavailable:", e)

AHS imports unavailable: No module named 'braket.ahs.program'


In [2]:
DEVICE_SPECS = {
    "aqt_ibex_q1": {
        "arn": "arn:aws:braket:eu-north-1::device/qpu/aqt/Ibex-Q1",
        "region": "eu-north-1",
        "provider": "AQT",
        "paradigm": "gate"
    },
    "ionq_forte_1": {
        "arn": "arn:aws:braket:us-east-1::device/qpu/ionq/Forte-1",
        "region": "us-east-1",
        "provider": "IonQ",
        "paradigm": "gate"
    },
    "iqm_garnet": {
        "arn": "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet",
        "region": "eu-north-1",
        "provider": "IQM",
        "paradigm": "gate"
    },
    "quera_aquila": {
        "arn": "arn:aws:braket:us-east-1::device/qpu/quera/Aquila",
        "region": "us-east-1",
        "provider": "QuEra",
        "paradigm": "ahs"
    },
    "rigetti_ankaa_3": {
        "arn": "arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3",
        "region": "us-west-1",
        "provider": "Rigetti",
        "paradigm": "gate"
    }
}

def load_devices(selected: Optional[Iterable[str]] = None) -> Dict[str, AwsDevice]:
    keys = list(selected) if selected is not None else list(DEVICE_SPECS.keys())
    return {k: AwsDevice(DEVICE_SPECS[k]["arn"]) for k in keys}

## Benchmark family

For the gate-model QPUs, this notebook uses **GHZ circuits** because their ideal output support is easy to compute:

\[
\mathrm{supp}_{ideal}(\mathrm{GHZ}_n) = \{0^n, 1^n\}
\]

This gives a natural empirical success metric:

\[
\mathrm{actual\_fidelity} = 
rac{\#\{	ext{shots in ideal support}\}}{	ext{total shots}}
\]

That matches your interpretation of fidelity as a success probability.

For QuEra Aquila, the notebook uses a small AHS benchmark where success is defined as landing in a target bitstring family.

In [ ]:
def ghz_circuit(n: int) -> Circuit:
    circ = Circuit().h(0)
    for q in range(n - 1):
        circ.cnot(q, q + 1)
    return circ

def ideal_support_ghz(n: int) -> Set[str]:
    return {"0"*n, "1"*n}

GATE_BENCHMARKS = [
    {"label": "ghz_2", "n_qubits": 2, "circuit": ghz_circuit(2), "ideal_support": ideal_support_ghz(2)},
    {"label": "ghz_4", "n_qubits": 4, "circuit": ghz_circuit(4), "ideal_support": ideal_support_ghz(4)},
    {"label": "ghz_7", "n_qubits": 7, "circuit": ghz_circuit(7), "ideal_support": ideal_support_ghz(7)},
    {"label": "ghz_10", "n_qubits": 10, "circuit": ghz_circuit(10), "ideal_support": ideal_support_ghz(10)},
]

In [4]:
@dataclass
class CanonicalCalibration:
    eps1q: Dict[Any, float]
    eps2q: Dict[Any, float]
    eps1q_avg: Optional[float]
    eps2q_avg: Optional[float]
    readout_fidelity: Dict[Any, float]
    readout_fidelity_avg: Optional[float]
    t1: Dict[Any, float]
    t2: Dict[Any, float]
    t1_avg: Optional[float]
    t2_avg: Optional[float]
    dur1q_ns: Dict[Any, int]
    dur2q_ns: Dict[Any, int]
    dur_meas_ns: Optional[int]
    dur1q_avg_ns: Optional[int]
    dur2q_avg_ns: Optional[int]
    init_survival_per_qubit: Optional[float] = None

def safe_get(obj, *paths, default=None):
    for path in paths:
        cur = obj
        try:
            keys = path.split(".") if isinstance(path, str) else list(path)
            ok = True
            for key in keys:
                if cur is None:
                    ok = False
                    break
                if isinstance(cur, dict):
                    cur = cur.get(key)
                else:
                    cur = getattr(cur, key)
            if ok and cur is not None:
                return cur
        except Exception:
            pass
    return default

def pct_to_prob(x):
    if x is None:
        return None
    return x / 100.0 if x > 1.0 else x

In [ ]:
# from statistics import mean

# def _safe_mean(xs):
#     xs = [x for x in xs if isinstance(x, (int, float)) and x > 0]
#     return mean(xs) if xs else None

# def _to_plain(obj):
#     if obj is None:
#         return None
#     if isinstance(obj, (str, int, float, bool)):
#         return obj
#     if isinstance(obj, dict):
#         return {str(k): _to_plain(v) for k, v in obj.items()}
#     if isinstance(obj, (list, tuple)):
#         return [_to_plain(v) for v in obj]
#     if hasattr(obj, "model_dump"):
#         try:
#             return _to_plain(obj.model_dump())
#         except Exception:
#             pass
#     if hasattr(obj, "dict"):
#         try:
#             return _to_plain(obj.dict())
#         except Exception:
#             pass
#     if hasattr(obj, "__dict__"):
#         return {
#             str(k): _to_plain(v)
#             for k, v in vars(obj).items()
#             if not k.startswith("_")
#         }
#     return obj

# def _find_fidelity_entry(entries, *, fidelity_types=None, gate_names=None):
#     if not entries:
#         return None

#     fidelity_types = fidelity_types or []
#     gate_names = gate_names or []

#     # First pass: match both gate + fidelity type if requested
#     for e in entries:
#         ft_name = ((e.get("fidelityType") or {}).get("name"))
#         g_name = e.get("gateName")
#         if gate_names and g_name not in gate_names:
#             continue
#         if fidelity_types and ft_name not in fidelity_types:
#             continue
#         val = e.get("fidelity")
#         if isinstance(val, (int, float)):
#             return float(val)

#     # Second pass: match gate only
#     if gate_names:
#         for e in entries:
#             g_name = e.get("gateName")
#             if g_name not in gate_names:
#                 continue
#             val = e.get("fidelity")
#             if isinstance(val, (int, float)):
#                 return float(val)

#     # Third pass: match fidelity type only
#     if fidelity_types:
#         for e in entries:
#             ft_name = ((e.get("fidelityType") or {}).get("name"))
#             if ft_name not in fidelity_types:
#                 continue
#             val = e.get("fidelity")
#             if isinstance(val, (int, float)):
#                 return float(val)

#     # Final fallback: first numeric fidelity
#     for e in entries:
#         val = e.get("fidelity")
#         if isinstance(val, (int, float)):
#             return float(val)

#     return None

# def _standardized_gate_model_dict(device: AwsDevice):
#     raw = getattr(device.properties, "standardized", device.properties)
#     return _to_plain(raw)

# def _normalize_standardized_gate_model(
#     device: AwsDevice,
#     *,
#     preferred_oneq_fidelity_types=None,
#     preferred_twoq_gate_names=None,
#     preferred_twoq_fidelity_types=None,
# ) -> CanonicalCalibration:
#     p = _standardized_gate_model_dict(device)

#     oneq = p.get("oneQubitProperties", {}) or {}
#     twoq = p.get("twoQubitProperties", {}) or {}

#     eps1q = {}
#     eps2q = {}
#     readout_fidelity = {}
#     t1 = {}
#     t2 = {}

#     preferred_oneq_fidelity_types = preferred_oneq_fidelity_types or [
#         "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#         "RANDOMIZED_BENCHMARKING",
#     ]
#     preferred_twoq_gate_names = preferred_twoq_gate_names or []
#     preferred_twoq_fidelity_types = preferred_twoq_fidelity_types or [
#         "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
#         "INTERLEAVED_RANDOMIZED_BENCHMARKING",
#         "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#         "RANDOMIZED_BENCHMARKING",
#     ]

#     for q, qprops in oneq.items():
#         q = str(q)

#         t1_val = ((qprops.get("T1") or {}).get("value"))
#         t2_val = ((qprops.get("T2") or {}).get("value"))

#         if isinstance(t1_val, (int, float)) and t1_val > 0:
#             t1[q] = float(t1_val)
#         if isinstance(t2_val, (int, float)) and t2_val > 0:
#             t2[q] = float(t2_val)

#         fentries = qprops.get("oneQubitFidelity", []) or []

#         f1 = _find_fidelity_entry(
#             fentries,
#             fidelity_types=preferred_oneq_fidelity_types,
#         )
#         if f1 is not None:
#             eps1q[q] = max(0.0, 1.0 - f1)

#         fro = _find_fidelity_entry(fentries, fidelity_types=["READOUT"])
#         if fro is None:
#             e01 = _find_fidelity_entry(fentries, fidelity_types=["READOUT_ERROR_0_TO_1"])
#             e10 = _find_fidelity_entry(fentries, fidelity_types=["READOUT_ERROR_1_TO_0"])
#             if e01 is not None and e10 is not None:
#                 fro = 1.0 - (e01 + e10) / 2.0

#         if fro is not None:
#             readout_fidelity[q] = fro

#     for edge, eprops in twoq.items():
#         edge = str(edge)
#         fentries = eprops.get("twoQubitGateFidelity", []) or []

#         f2 = _find_fidelity_entry(
#             fentries,
#             fidelity_types=preferred_twoq_fidelity_types,
#             gate_names=preferred_twoq_gate_names,
#         )
#         if f2 is not None:
#             eps2q[edge] = max(0.0, 1.0 - f2)

#     return CanonicalCalibration(
#         eps1q=eps1q,
#         eps2q=eps2q,
#         eps1q_avg=_safe_mean(list(eps1q.values())),
#         eps2q_avg=_safe_mean(list(eps2q.values())),
#         readout_fidelity=readout_fidelity,
#         readout_fidelity_avg=_safe_mean(list(readout_fidelity.values())),
#         t1=t1,
#         t2=t2,
#         t1_avg=_safe_mean(list(t1.values())),
#         t2_avg=_safe_mean(list(t2.values())),
#         dur1q_ns={},
#         dur2q_ns={},
#         dur_meas_ns=None,
#         dur1q_avg_ns=None,
#         dur2q_avg_ns=None,
#         init_survival_per_qubit=None,
#     )

# def normalize_iqm(device: AwsDevice) -> CanonicalCalibration:
#     return _normalize_standardized_gate_model(
#         device,
#         preferred_oneq_fidelity_types=[
#             "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#             "RANDOMIZED_BENCHMARKING",
#         ],
#         preferred_twoq_gate_names=["CZ"],
#         preferred_twoq_fidelity_types=[
#             "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
#             "INTERLEAVED_RANDOMIZED_BENCHMARKING",
#             "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#         ],
#     )

# def normalize_rigetti(device: AwsDevice) -> CanonicalCalibration:
#     return _normalize_standardized_gate_model(
#         device,
#         preferred_oneq_fidelity_types=[
#             "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#             "RANDOMIZED_BENCHMARKING",
#         ],
#         preferred_twoq_gate_names=[
#             "CZ",
#             "CPHASESHIFT",
#             "XY",
#             "ISWAP",
#             "CNOT",
#         ],
#         preferred_twoq_fidelity_types=[
#             "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
#             "INTERLEAVED_RANDOMIZED_BENCHMARKING",
#             "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
#             "RANDOMIZED_BENCHMARKING",
#         ],
#     )

# def normalize_ionq(device: AwsDevice) -> CanonicalCalibration:
#     p = device.properties
#     avg1 = pct_to_prob(
#         safe_get(p, "provider.fidelity.oneQubit", "provider.average.singleQubitFidelity", "provider.avg1qFidelityPct")
#     )
#     avg2 = pct_to_prob(
#         safe_get(p, "provider.fidelity.twoQubit", "provider.average.twoQubitFidelity", "provider.avg2qFidelityPct")
#     )
#     ro = pct_to_prob(
#         safe_get(p, "provider.fidelity.readout", "provider.average.readoutFidelity", "provider.avgReadoutFidelity")
#     )
#     t1 = safe_get(p, "provider.timing.t1", "provider.t1Seconds")
#     t2 = safe_get(p, "provider.timing.t2", "provider.t2Seconds")
#     d1 = safe_get(p, "provider.timing.oneQubitGate", "provider.oneQGateDurationSec")
#     d2 = safe_get(p, "provider.timing.twoQubitGate", "provider.twoQGateDurationSec")
#     dm = safe_get(p, "provider.timing.readout", "provider.readoutDurationSec")

#     return CanonicalCalibration(
#         eps1q={}, eps2q={},
#         eps1q_avg=(1.0 - avg1) if avg1 is not None else None,
#         eps2q_avg=(1.0 - avg2) if avg2 is not None else None,
#         readout_fidelity={}, readout_fidelity_avg=ro,
#         t1={}, t2={}, t1_avg=t1, t2_avg=t2,
#         dur1q_ns={}, dur2q_ns={},
#         dur_meas_ns=int(dm * 1e9) if dm is not None else None,
#         dur1q_avg_ns=int(d1 * 1e9) if d1 is not None else None,
#         dur2q_avg_ns=int(d2 * 1e9) if d2 is not None else None,
#     )

# def normalize_aqt(device: AwsDevice) -> CanonicalCalibration:
#     p = device.properties
#     f1 = pct_to_prob(safe_get(p, "provider.fidelity.oneQubit", "provider.oneQGateFidelity"))
#     f2 = pct_to_prob(safe_get(p, "provider.fidelity.twoQubit", "provider.twoQGateFidelity"))
#     ro = pct_to_prob(safe_get(p, "provider.fidelity.readout", "provider.readoutFidelity"))
#     t1 = safe_get(p, "provider.t1Seconds")
#     t2 = safe_get(p, "provider.t2Seconds")
#     d1 = safe_get(p, "provider.oneQGateDurationSec")
#     d2 = safe_get(p, "provider.twoQGateDurationSec")
#     dm = safe_get(p, "provider.readoutDurationSec")

#     return CanonicalCalibration(
#         eps1q={}, eps2q={},
#         eps1q_avg=(1.0 - f1) if f1 is not None else None,
#         eps2q_avg=(1.0 - f2) if f2 is not None else None,
#         readout_fidelity={}, readout_fidelity_avg=ro,
#         t1={}, t2={}, t1_avg=t1, t2_avg=t2,
#         dur1q_ns={}, dur2q_ns={},
#         dur_meas_ns=int(dm * 1e9) if dm is not None else None,
#         dur1q_avg_ns=int(d1 * 1e9) if d1 is not None else None,
#         dur2q_avg_ns=int(d2 * 1e9) if d2 is not None else None,
#     )

# def normalize_calibration(device_key: str, device: AwsDevice) -> CanonicalCalibration:
#     provider = DEVICE_SPECS[device_key]["provider"]
#     if provider == "IonQ":
#         return normalize_ionq(device)
#     if provider == "AQT":
#         return normalize_aqt(device)
#     if provider == "Rigetti":
#         return normalize_rigetti(device)
#     if provider == "IQM":
#         return normalize_iqm(device)
#     if provider == "QuEra":
#         return normalize_iqm(device)  # temporary placeholder until separate AHS normalization is restored
#     raise ValueError(f"Unsupported provider: {provider}")

from statistics import mean

def _safe_mean(xs):
    xs = [x for x in xs if x is not None and isinstance(x, (int, float)) and x > 0]
    return mean(xs) if xs else None

def _to_plain(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): _to_plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_plain(v) for v in obj]
    if hasattr(obj, "model_dump"):
        try:
            return _to_plain(obj.model_dump())
        except Exception:
            pass
    if hasattr(obj, "dict"):
        try:
            return _to_plain(obj.dict())
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        return {
            str(k): _to_plain(v)
            for k, v in vars(obj).items()
            if not k.startswith("_")
        }
    return obj

def _find_fidelity_entry(entries, *, fidelity_types=None, gate_names=None):
    if not entries:
        return None

    fidelity_types = fidelity_types or []
    gate_names = gate_names or []

    for e in entries:
        ft_name = ((e.get("fidelityType") or {}).get("name"))
        g_name = e.get("gateName")
        if gate_names and g_name not in gate_names:
            continue
        if fidelity_types and ft_name not in fidelity_types:
            continue
        val = e.get("fidelity")
        if isinstance(val, (int, float)):
            return float(val)

    if gate_names:
        for e in entries:
            g_name = e.get("gateName")
            if g_name not in gate_names:
                continue
            val = e.get("fidelity")
            if isinstance(val, (int, float)):
                return float(val)

    if fidelity_types:
        for e in entries:
            ft_name = ((e.get("fidelityType") or {}).get("name"))
            if ft_name not in fidelity_types:
                continue
            val = e.get("fidelity")
            if isinstance(val, (int, float)):
                return float(val)

    for e in entries:
        val = e.get("fidelity")
        if isinstance(val, (int, float)):
            return float(val)

    return None

def _normalize_standardized_gate_model(
    device: AwsDevice,
    *,
    preferred_oneq_fidelity_types=None,
    preferred_twoq_gate_names=None,
    preferred_twoq_fidelity_types=None,
) -> CanonicalCalibration:
    raw = getattr(device.properties, "standardized", device.properties)
    p = _to_plain(raw)

    oneq = p.get("oneQubitProperties", {}) or {}
    twoq = p.get("twoQubitProperties", {}) or {}

    eps1q = {}
    eps2q = {}
    readout_fidelity = {}
    t1 = {}
    t2 = {}

    preferred_oneq_fidelity_types = preferred_oneq_fidelity_types or [
        "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
        "RANDOMIZED_BENCHMARKING",
    ]
    preferred_twoq_gate_names = preferred_twoq_gate_names or []
    preferred_twoq_fidelity_types = preferred_twoq_fidelity_types or [
        "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
        "INTERLEAVED_RANDOMIZED_BENCHMARKING",
        "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
        "RANDOMIZED_BENCHMARKING",
    ]

    for q, qprops in oneq.items():
        q = str(q)

        t1_val = ((qprops.get("T1") or {}).get("value"))
        t2_val = ((qprops.get("T2") or {}).get("value"))

        if isinstance(t1_val, (int, float)) and t1_val > 0:
            t1[q] = float(t1_val)
        if isinstance(t2_val, (int, float)) and t2_val > 0:
            t2[q] = float(t2_val)

        fentries = qprops.get("oneQubitFidelity", []) or []

        f1 = _find_fidelity_entry(
            fentries,
            fidelity_types=preferred_oneq_fidelity_types,
        )
        if f1 is not None:
            eps1q[q] = max(0.0, 1.0 - f1)

        fro = _find_fidelity_entry(fentries, fidelity_types=["READOUT"])
        if fro is None:
            e01 = _find_fidelity_entry(fentries, fidelity_types=["READOUT_ERROR_0_TO_1"])
            e10 = _find_fidelity_entry(fentries, fidelity_types=["READOUT_ERROR_1_TO_0"])
            if e01 is not None and e10 is not None:
                fro = 1.0 - (e01 + e10) / 2.0

        if fro is not None:
            readout_fidelity[q] = fro

    for edge, eprops in twoq.items():
        edge = str(edge)
        fentries = eprops.get("twoQubitGateFidelity", []) or []

        f2 = _find_fidelity_entry(
            fentries,
            fidelity_types=preferred_twoq_fidelity_types,
            gate_names=preferred_twoq_gate_names,
        )
        if f2 is not None:
            eps2q[edge] = max(0.0, 1.0 - f2)

    return CanonicalCalibration(
        eps1q=eps1q,
        eps2q=eps2q,
        eps1q_avg=_safe_mean(list(eps1q.values())),
        eps2q_avg=_safe_mean(list(eps2q.values())),
        readout_fidelity=readout_fidelity,
        readout_fidelity_avg=_safe_mean(list(readout_fidelity.values())),
        t1=t1,
        t2=t2,
        t1_avg=_safe_mean(list(t1.values())),
        t2_avg=_safe_mean(list(t2.values())),
        dur1q_ns={},
        dur2q_ns={},
        dur_meas_ns=None,
        dur1q_avg_ns=None,
        dur2q_avg_ns=None,
        init_survival_per_qubit=None,
    )

def normalize_iqm(device: AwsDevice) -> CanonicalCalibration:
    return _normalize_standardized_gate_model(
        device,
        preferred_oneq_fidelity_types=[
            "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
            "RANDOMIZED_BENCHMARKING",
        ],
        preferred_twoq_gate_names=["CZ"],
        preferred_twoq_fidelity_types=[
            "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
            "INTERLEAVED_RANDOMIZED_BENCHMARKING",
            "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
        ],
    )

def normalize_rigetti(device: AwsDevice) -> CanonicalCalibration:
    return _normalize_standardized_gate_model(
        device,
        preferred_oneq_fidelity_types=[
            "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
            "RANDOMIZED_BENCHMARKING",
        ],
        preferred_twoq_gate_names=[
            "ISWAP",
            "XY",
            "CZ",
            "CPHASESHIFT",
            "CNOT",
        ],
        preferred_twoq_fidelity_types=[
            "SIMULTANEOUS_INTERLEAVED_RANDOMIZED_BENCHMARKING",
            "INTERLEAVED_RANDOMIZED_BENCHMARKING",
            "SIMULTANEOUS_RANDOMIZED_BENCHMARKING",
            "RANDOMIZED_BENCHMARKING",
        ],
    )

def normalize_ionq(device: AwsDevice) -> CanonicalCalibration:
    p = device.properties
    avg1 = pct_to_prob(
        safe_get(p, "provider.fidelity.oneQubit", "provider.average.singleQubitFidelity", "provider.avg1qFidelityPct")
    )
    avg2 = pct_to_prob(
        safe_get(p, "provider.fidelity.twoQubit", "provider.average.twoQubitFidelity", "provider.avg2qFidelityPct")
    )
    ro = pct_to_prob(
        safe_get(p, "provider.fidelity.readout", "provider.average.readoutFidelity", "provider.avgReadoutFidelity")
    )
    t1 = safe_get(p, "provider.timing.t1", "provider.t1Seconds")
    t2 = safe_get(p, "provider.timing.t2", "provider.t2Seconds")
    d1 = safe_get(p, "provider.timing.oneQubitGate", "provider.oneQGateDurationSec")
    d2 = safe_get(p, "provider.timing.twoQubitGate", "provider.twoQGateDurationSec")
    dm = safe_get(p, "provider.timing.readout", "provider.readoutDurationSec")

    return CanonicalCalibration(
        eps1q={}, eps2q={},
        eps1q_avg=(1.0 - avg1) if avg1 is not None else None,
        eps2q_avg=(1.0 - avg2) if avg2 is not None else None,
        readout_fidelity={}, readout_fidelity_avg=ro,
        t1={}, t2={}, t1_avg=t1, t2_avg=t2,
        dur1q_ns={}, dur2q_ns={},
        dur_meas_ns=int(dm * 1e9) if dm is not None else None,
        dur1q_avg_ns=int(d1 * 1e9) if d1 is not None else None,
        dur2q_avg_ns=int(d2 * 1e9) if d2 is not None else None,
    )

def normalize_aqt(device: AwsDevice) -> CanonicalCalibration:
    p = device.properties
    f1 = pct_to_prob(safe_get(p, "provider.fidelity.oneQubit", "provider.oneQGateFidelity"))
    f2 = pct_to_prob(safe_get(p, "provider.fidelity.twoQubit", "provider.twoQGateFidelity"))
    ro = pct_to_prob(safe_get(p, "provider.fidelity.readout", "provider.readoutFidelity"))
    t1 = safe_get(p, "provider.t1Seconds")
    t2 = safe_get(p, "provider.t2Seconds")
    d1 = safe_get(p, "provider.oneQGateDurationSec")
    d2 = safe_get(p, "provider.twoQGateDurationSec")
    dm = safe_get(p, "provider.readoutDurationSec")

    return CanonicalCalibration(
        eps1q={}, eps2q={},
        eps1q_avg=(1.0 - f1) if f1 is not None else None,
        eps2q_avg=(1.0 - f2) if f2 is not None else None,
        readout_fidelity={}, readout_fidelity_avg=ro,
        t1={}, t2={}, t1_avg=t1, t2_avg=t2,
        dur1q_ns={}, dur2q_ns={},
        dur_meas_ns=int(dm * 1e9) if dm is not None else None,
        dur1q_avg_ns=int(d1 * 1e9) if d1 is not None else None,
        dur2q_avg_ns=int(d2 * 1e9) if d2 is not None else None,
    )

def normalize_calibration(device_key: str, device: AwsDevice) -> CanonicalCalibration:
    provider = DEVICE_SPECS[device_key]["provider"]
    if provider == "IonQ":
        return normalize_ionq(device)
    if provider == "AQT":
        return normalize_aqt(device)
    if provider == "Rigetti":
        return normalize_rigetti(device)
    if provider == "IQM":
        return normalize_iqm(device)
    if provider == "QuEra":
        return normalize_iqm(device)  
    raise ValueError(f"Unsupported provider: {provider}")

In [6]:
devices = load_devices()
iqm_cal = normalize_calibration("iqm_garnet", devices["iqm_garnet"])
print("eps1q count:", len(iqm_cal.eps1q))
print("eps2q count:", len(iqm_cal.eps2q))
print("readout count:", len(iqm_cal.readout_fidelity))
print("t1 count:", len(iqm_cal.t1))
print("t2 count:", len(iqm_cal.t2))
print("eps1q_avg:", iqm_cal.eps1q_avg)
print("eps2q_avg:", iqm_cal.eps2q_avg)
print("readout_fidelity_avg:", iqm_cal.readout_fidelity_avg)
print("t2_avg:", iqm_cal.t2_avg)

eps1q count: 20
eps2q count: 30
readout count: 20
t1 count: 20
t2 count: 19
eps1q_avg: 0.0011576754993965987
eps2q_avg: 0.007641002491395806
readout_fidelity_avg: 0.9801
t2_avg: 9.302200712200168e-06


In [ ]:
iqm_cal = normalize_calibration("rigetti_ankaa_3", devices["rigetti_ankaa_3"])
print("eps1q count:", len(iqm_cal.eps1q))
print("eps2q count:", len(iqm_cal.eps2q))
print("readout count:", len(iqm_cal.readout_fidelity))
print("t1 count:", len(iqm_cal.t1))
print("t2 count:", len(iqm_cal.t2))
print("eps1q_avg:", iqm_cal.eps1q_avg)
print("eps2q_avg:", iqm_cal.eps2q_avg)
print("readout_fidelity_avg:", iqm_cal.readout_fidelity_avg)
print("t2_avg:", iqm_cal.t2_avg)

In [ ]:
# def count_ops(circuit: Circuit) -> Tuple[int, int]:
#     one_q = 0
#     two_q = 0
#     for instr in circuit.instructions:
#         nq = len(instr.target)
#         if nq == 1:
#             one_q += 1
#         elif nq == 2:
#             two_q += 1
#     return one_q, two_q

# def approximate_per_qubit_time_ns(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int) -> float:
#     n1, n2 = count_ops(circuit)
#     d1 = cal.dur1q_avg_ns or 0
#     d2 = cal.dur2q_avg_ns or 0
#     dm = cal.dur_meas_ns or 0
#     total_gate_time_ns = n1 * d1 + n2 * d2
#     return total_gate_time_ns / max(n_qubits, 1) + dm

# def estimated_success_probability(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int):
#     n1, n2 = count_ops(circuit)
#     used_any_term = False
#     log_p_total = 0.0

#     if cal.eps1q_avg is not None:
#         log_p_total += n1 * math.log(max(1e-12, 1.0 - cal.eps1q_avg))
#         used_any_term = True

#     if cal.eps2q_avg is not None:
#         log_p_total += n2 * math.log(max(1e-12, 1.0 - cal.eps2q_avg))
#         used_any_term = True

#     if cal.readout_fidelity_avg is not None:
#         log_p_total += n_qubits * math.log(max(1e-12, cal.readout_fidelity_avg))
#         used_any_term = True

#     if cal.t2_avg is not None:
#         t_q_ns = approximate_per_qubit_time_ns(circuit, cal, n_qubits)
#         t_q_sec = t_q_ns * 1e-9
#         log_p_total += n_qubits * (-t_q_sec / max(cal.t2_avg, 1e-12))
#         used_any_term = True

#     if cal.init_survival_per_qubit is not None:
#         log_p_total += n_qubits * math.log(max(1e-12, cal.init_survival_per_qubit))
#         used_any_term = True

#     if not used_any_term:
#         return None

#     return math.exp(log_p_total)

#######################################################################
# def count_ops(circuit: Circuit) -> Tuple[int, int]:
#     one_q = 0
#     two_q = 0
#     for instr in circuit.instructions:
#         nq = len(instr.target)
#         if nq == 1:
#             one_q += 1
#         elif nq == 2:
#             two_q += 1
#     return one_q, two_q

# def _targets_as_strs(instr):
#     return [str(int(q)) for q in list(instr.target)]

# def _edge_candidates(a: str, b: str):
#     return [f"{a}-{b}", f"{b}-{a}", f"{min(int(a), int(b))}-{max(int(a), int(b))}"]

# def _lookup_edge_value(edge_map: Dict[Any, float], a: str, b: str):
#     for k in _edge_candidates(a, b):
#         if k in edge_map:
#             return edge_map[k]
#     return None

# def approximate_per_qubit_time_ns(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int) -> float:
#     n1, n2 = count_ops(circuit)
#     d1 = cal.dur1q_avg_ns or 0
#     d2 = cal.dur2q_avg_ns or 0
#     dm = cal.dur_meas_ns or 0
#     total_gate_time_ns = n1 * d1 + n2 * d2
#     return total_gate_time_ns / max(n_qubits, 1) + dm

# def estimated_success_probability(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int):
#     used_any_term = False
#     log_p_total = 0.0
#     used_qubits = set()

#     # Per-instruction operational penalties
#     for instr in circuit.instructions:
#         targets = _targets_as_strs(instr)

#         if len(targets) == 1:
#             q = targets[0]
#             used_qubits.add(q)

#             eps = cal.eps1q.get(q)
#             if eps is None:
#                 eps = cal.eps1q_avg

#             if eps is not None:
#                 log_p_total += math.log(max(1e-12, 1.0 - eps))
#                 used_any_term = True

#         elif len(targets) == 2:
#             a, b = targets
#             used_qubits.add(a)
#             used_qubits.add(b)

#             eps = _lookup_edge_value(cal.eps2q, a, b)
#             if eps is None:
#                 eps = cal.eps2q_avg

#             if eps is not None:
#                 log_p_total += math.log(max(1e-12, 1.0 - eps))
#                 used_any_term = True

#     # Readout penalty on all qubits touched by the circuit
#     if used_qubits:
#         for q in used_qubits:
#             fro = cal.readout_fidelity.get(q)
#             if fro is None:
#                 fro = cal.readout_fidelity_avg

#             if fro is not None:
#                 log_p_total += math.log(max(1e-12, fro))
#                 used_any_term = True
#     elif cal.readout_fidelity_avg is not None:
#         log_p_total += n_qubits * math.log(max(1e-12, cal.readout_fidelity_avg))
#         used_any_term = True

#     # Decoherence: use per-qubit T2 when present, else average
#     t_q_ns = approximate_per_qubit_time_ns(circuit, cal, n_qubits)
#     t_q_sec = t_q_ns * 1e-9

#     if used_qubits:
#         for q in used_qubits:
#             t2q = cal.t2.get(q)
#             if t2q is None:
#                 t2q = cal.t2_avg
#             if t2q is not None:
#                 log_p_total += -t_q_sec / max(t2q, 1e-12)
#                 used_any_term = True
#     elif cal.t2_avg is not None:
#         log_p_total += n_qubits * (-t_q_sec / max(cal.t2_avg, 1e-12))
#         used_any_term = True

#     # Neutral-atom initialization / survival term
#     if cal.init_survival_per_qubit is not None:
#         nq = len(used_qubits) if used_qubits else n_qubits
#         log_p_total += nq * math.log(max(1e-12, cal.init_survival_per_qubit))
#         used_any_term = True

#     if not used_any_term:
#         return None

#     return math.exp(log_p_total)

import re
from collections import Counter, defaultdict

def count_ops(circuit: Circuit) -> Tuple[int, int]:
    one_q = 0
    two_q = 0
    for instr in circuit.instructions:
        nq = len(instr.target)
        if nq == 1:
            one_q += 1
        elif nq == 2:
            two_q += 1
    return one_q, two_q

def _targets_as_strs(instr):
    return [str(int(q)) for q in list(instr.target)]

def _edge_candidates(a: str, b: str):
    return [f"{a}-{b}", f"{b}-{a}", f"{min(int(a), int(b))}-{max(int(a), int(b))}"]

def _lookup_edge_value(edge_map: Dict[Any, float], a: str, b: str):
    for k in _edge_candidates(a, b):
        if k in edge_map:
            return edge_map[k]
    return None

def approximate_per_qubit_time_ns(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int) -> float:
    n1, n2 = count_ops(circuit)
    d1 = cal.dur1q_avg_ns or 0
    d2 = cal.dur2q_avg_ns or 0
    dm = cal.dur_meas_ns or 0
    total_gate_time_ns = n1 * d1 + n2 * d2
    return total_gate_time_ns / max(n_qubits, 1) + dm

def estimated_success_probability(circuit: Circuit, cal: CanonicalCalibration, n_qubits: int):
    used_any_term = False
    log_p_total = 0.0
    used_qubits = set()

    for instr in circuit.instructions:
        targets = _targets_as_strs(instr)

        if len(targets) == 1:
            q = targets[0]
            used_qubits.add(q)

            eps = cal.eps1q.get(q)
            if eps is None:
                eps = cal.eps1q_avg

            if eps is not None:
                log_p_total += math.log(max(1e-12, 1.0 - eps))
                used_any_term = True

        elif len(targets) == 2:
            a, b = targets
            used_qubits.add(a)
            used_qubits.add(b)

            eps = _lookup_edge_value(cal.eps2q, a, b)
            if eps is None:
                eps = cal.eps2q_avg

            if eps is not None:
                log_p_total += math.log(max(1e-12, 1.0 - eps))
                used_any_term = True

    if used_qubits:
        for q in used_qubits:
            fro = cal.readout_fidelity.get(q)
            if fro is None:
                fro = cal.readout_fidelity_avg

            if fro is not None:
                log_p_total += math.log(max(1e-12, fro))
                used_any_term = True
    elif cal.readout_fidelity_avg is not None:
        log_p_total += n_qubits * math.log(max(1e-12, cal.readout_fidelity_avg))
        used_any_term = True

    t_q_ns = approximate_per_qubit_time_ns(circuit, cal, n_qubits)
    t_q_sec = t_q_ns * 1e-9

    if used_qubits:
        for q in used_qubits:
            t2q = cal.t2.get(q)
            if t2q is None:
                t2q = cal.t2_avg
            if t2q is not None:
                log_p_total += -t_q_sec / max(t2q, 1e-12)
                used_any_term = True
    elif cal.t2_avg is not None:
        log_p_total += n_qubits * (-t_q_sec / max(cal.t2_avg, 1e-12))
        used_any_term = True

    if cal.init_survival_per_qubit is not None:
        nq = len(used_qubits) if used_qubits else n_qubits
        log_p_total += nq * math.log(max(1e-12, cal.init_survival_per_qubit))
        used_any_term = True

    if not used_any_term:
        return None

    return math.exp(log_p_total)

ZERO_COST_NATIVE_1Q_GATES = {"RZ", "PHASE", "PHASESHIFT"}
COUNTED_NATIVE_1Q_GATES = {"RX", "X", "Y", "H"}
COUNTED_NATIVE_2Q_GATES = {"ISWAP", "XY", "CZ", "CPHASESHIFT", "CNOT"}

def _canonical_edge(a: str, b: str) -> str:
    return f"{min(int(a), int(b))}-{max(int(a), int(b))}"

def parse_rigetti_compiled_program(compiled_program: str):
    oneq_counts = Counter()
    twoq_counts = Counter()
    measured_qubits = []
    used_qubits = set()

    for raw in compiled_program.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith("PRAGMA") or line.startswith("DECLARE"):
            continue

        m_meas = re.match(r"MEASURE\s+(\d+)\s+ro\[\d+\]", line)
        if m_meas:
            q = m_meas.group(1)
            measured_qubits.append(q)
            used_qubits.add(q)
            continue

        m_gate = re.match(r"([A-Z_]+)(?:\([^)]*\))?\s+(\d+)(?:\s+(\d+))?$", line)
        if not m_gate:
            continue

        gate = m_gate.group(1)
        q1 = m_gate.group(2)
        q2 = m_gate.group(3)

        used_qubits.add(q1)

        if q2 is None:
            if gate in COUNTED_NATIVE_1Q_GATES:
                oneq_counts[q1] += 1
            elif gate in ZERO_COST_NATIVE_1Q_GATES:
                pass
            else:
                oneq_counts[q1] += 1
        else:
            used_qubits.add(q2)
            if gate in COUNTED_NATIVE_2Q_GATES:
                edge = _canonical_edge(q1, q2)
                twoq_counts[edge] += 1
            else:
                edge = _canonical_edge(q1, q2)
                twoq_counts[edge] += 1

    return {
        "oneq_counts": dict(oneq_counts),
        "twoq_counts": dict(twoq_counts),
        "measured_qubits": measured_qubits,
        "used_qubits": sorted(used_qubits, key=lambda x: int(x)),
    }

def estimate_rigetti_from_compiled_program(
    compiled_program: str,
    cal: CanonicalCalibration,
    include_decoherence: bool = False,
) -> Optional[float]:
    parsed = parse_rigetti_compiled_program(compiled_program)

    oneq_counts = parsed["oneq_counts"]
    twoq_counts = parsed["twoq_counts"]
    measured_qubits = parsed["measured_qubits"]
    used_qubits = parsed["used_qubits"]

    log_p_total = 0.0
    used_any_term = False

    for q, cnt in oneq_counts.items():
        eps = cal.eps1q.get(q)
        if eps is None:
            eps = cal.eps1q_avg
        if eps is not None:
            log_p_total += cnt * math.log(max(1e-12, 1.0 - eps))
            used_any_term = True

    for edge, cnt in twoq_counts.items():
        eps = cal.eps2q.get(edge)
        if eps is None:
            eps = cal.eps2q_avg
        if eps is not None:
            log_p_total += cnt * math.log(max(1e-12, 1.0 - eps))
            used_any_term = True

    for q in measured_qubits:
        fro = cal.readout_fidelity.get(q)
        if fro is None:
            fro = cal.readout_fidelity_avg
        if fro is not None:
            log_p_total += math.log(max(1e-12, fro))
            used_any_term = True

    if include_decoherence:
        pass

    if not used_any_term:
        return None

    return math.exp(log_p_total)

In [8]:
def result_counts(result) -> Dict[str, int]:
    if hasattr(result, "measurement_counts") and result.measurement_counts is not None:
        return dict(result.measurement_counts)
    if hasattr(result, "get_counts"):
        return dict(result.get_counts())
    if hasattr(result, "measurements") and result.measurements is not None:
        counts = {}
        for row in result.measurements:
            bitstr = "".join(str(int(x)) for x in row)
            counts[bitstr] = counts.get(bitstr, 0) + 1
        return counts
    raise ValueError("Could not extract measurement counts from task result.")

def empirical_success_from_counts(counts: Dict[str, int], ideal_support: Set[str]) -> float:
    total = sum(counts.values())
    if total == 0:
        return float("nan")
    success = sum(v for k, v in counts.items() if k in ideal_support)
    return success / total

In [9]:
# import time

# TERMINAL_STATES = {"COMPLETED", "FAILED", "CANCELLED"}
# RESULTS_READY_STATES = {"COMPLETED"}

# def wait_for_task_terminal(task, poll_seconds: float = 2.0, verbose: bool = True):
#     while True:
#         status = task.state()
#         if verbose:
#             print(f"Task {task.id} status: {status}")
#         if status in TERMINAL_STATES:
#             return status
#         time.sleep(poll_seconds)

# def run_gate_benchmark(device: AwsDevice, circuit: Circuit, shots: int = 200):
#     task = device.run(circuit, shots=shots)
#     print("Submitted:", task.id)

#     status = wait_for_task_terminal(task, poll_seconds=2.0, verbose=False)

#     if status not in RESULTS_READY_STATES:
#         raise RuntimeError(f"Task ended in terminal state {status}")

#     # Now this should take the fast path in result() and download the result,
#     # instead of trying to drive asyncio polling from inside the notebook.
#     result = task.result()
#     return task, result

# def benchmark_gate_device(device_key: str, benchmark: dict, shots: int = 200) -> dict:
#     device = devices[device_key]
#     cal = normalize_calibration(device_key, device)
#     # print(cal) #PRINT CALIBRATION
#     est = estimated_success_probability(benchmark["circuit"], cal, benchmark["n_qubits"])
#     task, result = run_gate_benchmark(device, benchmark["circuit"], shots=shots)
#     compiled = result.get_compiled_circuit()
#     print(compiled)
#     counts = result_counts(result)
#     act = empirical_success_from_counts(counts, benchmark["ideal_support"])
#     return {
#         "device_key": device_key,
#         "provider": DEVICE_SPECS[device_key]["provider"],
#         "benchmark": benchmark["label"],
#         "n_qubits": benchmark["n_qubits"],
#         "estimated_fidelity": est,
#         "actual_fidelity": act,
#         "skew_abs": abs(act - est) if est is not None else None,
#         "task_id": task.id,
#         "counts": counts,
#     }

import time

TERMINAL_STATES = {"COMPLETED", "FAILED", "CANCELLED"}
RESULTS_READY_STATES = {"COMPLETED"}

def wait_for_task_terminal(task, poll_seconds: float = 2.0, verbose: bool = True):
    while True:
        status = task.state()
        if verbose:
            print(f"Task {task.id} status: {status}")
        if status in TERMINAL_STATES:
            return status
        time.sleep(poll_seconds)

def run_gate_benchmark(device: AwsDevice, circuit: Circuit, shots: int = 200):
    task = device.run(circuit, shots=shots)
    print("Submitted:", task.id)

    status = wait_for_task_terminal(task, poll_seconds=2.0, verbose=False)

    if status not in RESULTS_READY_STATES:
        raise RuntimeError(f"Task ended in terminal state {status}")

    result = task.result()
    return task, result

def get_compiled_program(result):
    getters = [
        lambda r: r.get_compiled_circuit(),
        lambda r: r.additional_metadata.rigettiMetadata.compiledProgram,
        lambda r: r.additional_metadata.iqmMetadata.compiledProgram,
    ]
    for g in getters:
        try:
            v = g(result)
            if v:
                return v
        except Exception:
            pass
    return None

def benchmark_gate_device(device_key: str, benchmark: dict, shots: int = 200) -> dict:
    device = devices[device_key]
    provider = DEVICE_SPECS[device_key]["provider"]
    cal = normalize_calibration(device_key, device)

    logical_est = estimated_success_probability(
        benchmark["circuit"], cal, benchmark["n_qubits"]
    )

    task, result = run_gate_benchmark(device, benchmark["circuit"], shots=shots)
    compiled_program = get_compiled_program(result)

    compiled_est = None
    est = logical_est

    if provider == "Rigetti" and compiled_program:
        compiled_est = estimate_rigetti_from_compiled_program(
            compiled_program,
            cal,
            include_decoherence=False,
        )
        est = compiled_est

    counts = result_counts(result)
    act = empirical_success_from_counts(counts, benchmark["ideal_support"])

    return {
        "device_key": device_key,
        "provider": provider,
        "benchmark": benchmark["label"],
        "n_qubits": benchmark["n_qubits"],
        "estimated_fidelity": est,
        "estimated_fidelity_logical": logical_est,
        "estimated_fidelity_compiled": compiled_est,
        "actual_fidelity": act,
        "skew_abs": abs(act - est) if est is not None else None,
        "task_id": task.id,
        "counts": counts,
        "compiled_program": compiled_program,
    }

## Gate-model submissions

Per your request, **AQT** and **IonQ** submission blocks are commented out.

You can run IQM and Rigetti directly when those QPUs are available to your account.

In [10]:
devices = load_devices()

gate_results = []

for b in GATE_BENCHMARKS:
    # try:
    #     row = benchmark_gate_device("iqm_garnet", b, shots=200)
    #     gate_results.append(row)
    #     print("Completed:", row["provider"], row["benchmark"], row["actual_fidelity"])
    # except Exception as e:
    #     print("IQM failed on", b["label"], "->", e)

    try:
        row = benchmark_gate_device("rigetti_ankaa_3", b, shots=200)
        gate_results.append(row)
        print("Completed:", row["provider"], row["benchmark"], row["actual_fidelity"])
    except Exception as e:
        print("Rigetti failed on", b["label"], "->", e)

    # try:
    #     row = benchmark_gate_device("ionq_forte_1", b, shots=200)
    #     gate_results.append(row)
    #     print("Completed:", row["provider"], row["benchmark"], row["actual_fidelity"])
    # except Exception as e:
    #     print("IonQ failed on", b["label"], "->", e)

    # try:
    #     row = benchmark_gate_device("aqt_ibex_q1", b, shots=200)
    #     gate_results.append(row)
    #     print("Completed:", row["provider"], row["benchmark"], row["actual_fidelity"])
    # except Exception as e:
    #     print("AQT failed on", b["label"], "->", e)

gate_df = pd.DataFrame(gate_results)
gate_df

Submitted: arn:aws:braket:us-west-1:963294351349:quantum-task/591cbfe2-b8c5-4276-a44f-f29005169e52
Completed: Rigetti ghz_2 0.875
Submitted: arn:aws:braket:us-west-1:963294351349:quantum-task/1434642c-b913-4539-b44c-c85d013df2eb
Completed: Rigetti ghz_4 0.835
Submitted: arn:aws:braket:us-west-1:963294351349:quantum-task/819f703b-2867-4857-863a-1de6ef31f440
Completed: Rigetti ghz_7 0.63
Submitted: arn:aws:braket:us-west-1:963294351349:quantum-task/b91dcd18-9a75-4d5f-b291-6be08a88ef97
Completed: Rigetti ghz_10 0.615


,device_key,provider,benchmark,n_qubits,estimated_fidelity,estimated_fidelity_logical,estimated_fidelity_compiled,actual_fidelity,skew_abs,task_id,counts,compiled_program
0,rigetti_ankaa_3,Rigetti,ghz_2,2,0.924783,0.879990,0.924783,0.875,0.049783,arn:aws:braket:us-west-1:963294351349:quantum-...,"{'00': 82, '11': 93, '01': 13, '10': 12}","PRAGMA INITIAL_REWIRING ""NAIVE""\nDECLARE ro BI..."
1,rigetti_ankaa_3,Rigetti,ghz_4,4,0.881437,0.813254,0.881437,0.835,0.046437,arn:aws:braket:us-west-1:963294351349:quantum-...,"{'1111': 77, '0000': 90, '1000': 9, '0010': 1,...","PRAGMA INITIAL_REWIRING ""NAIVE""\nDECLARE ro BI..."
2,rigetti_ankaa_3,Rigetti,ghz_7,7,0.724713,0.183035,0.724713,0.630,0.094713,arn:aws:braket:us-west-1:963294351349:quantum-...,"{'1011111': 4, '0000000': 63, '1111111': 63, '...","PRAGMA INITIAL_REWIRING ""NAIVE""\nDECLARE ro BI..."
3,rigetti_ankaa_3,Rigetti,ghz_10,10,0.643497,0.153170,0.643497,0.615,0.028497,arn:aws:braket:us-west-1:963294351349:quantum-...,"{'0000000000': 70, '1111111111': 53, '11100000...","PRAGMA INITIAL_REWIRING ""NAIVE""\nDECLARE ro BI..."


## QuEra Aquila benchmark

Aquila is an **AHS** device, so the notebook cannot submit the GHZ circuits there. Instead, this section defines a small neutral-atom benchmark where:

- `estimated_fidelity` uses the neutral-atom terms from your model (`P_fill`, `P_loss`, readout fidelity if exposed)
- `actual_fidelity` is the empirical success probability of landing in a target bitstring family

The code below is a scaffold. In practice, you should inspect one Aquila result and adapt the extraction code if your SDK version formats AHS measurements differently.

In [ ]:
def quera_estimated_success_from_calibration(device_key: str, n_atoms: int) -> Optional[float]:
    device = devices[device_key]
    cal = normalize_calibration(device_key, device)
    log_p = 0.0
    if cal.init_survival_per_qubit is not None:
        log_p += n_atoms * math.log(max(1e-12, cal.init_survival_per_qubit))
    if cal.readout_fidelity_avg is not None:
        log_p += n_atoms * math.log(max(1e-12, cal.readout_fidelity_avg))
    return math.exp(log_p)

def target_family_checkerboard(n_atoms: int) -> Set[str]:
    a = "".join("1" if i % 2 == 0 else "0" for i in range(n_atoms))
    b = "".join("0" if i % 2 == 0 else "1" for i in range(n_atoms))
    return {a, b}

def ahs_result_to_counts(result) -> Dict[str, int]:
    if hasattr(result, "measurements") and result.measurements is not None:
        counts = {}
        for m in result.measurements:
            if hasattr(m, "post_sequence"):
                bitstr = "".join("1" if x else "0" for x in m.post_sequence)
            else:
                bitstr = str(m)
            counts[bitstr] = counts.get(bitstr, 0) + 1
        return counts
    raise ValueError("Inspect one Aquila result and adapt ahs_result_to_counts().")

def empirical_ahs_success(result, target_family: Set[str]) -> float:
    counts = ahs_result_to_counts(result)
    return empirical_success_from_counts(counts, target_family)

def build_minimal_quera_program(n_atoms: int):
    if not AHS_AVAILABLE:
        raise RuntimeError("AHS classes are not available in this notebook environment.")

    register = AtomArrangement()
    spacing = 6.1e-6
    for i in range(n_atoms):
        register.add((i * spacing, 0.0))

    omega = TimeSeries().put(0.0, 0.0).put(1e-6, 1.5e7).put(2e-6, 1.5e7).put(3e-6, 0.0)
    detuning = TimeSeries().put(0.0, -1.0e7).put(3e-6, 1.0e7)
    phase = TimeSeries().put(0.0, 0.0).put(3e-6, 0.0)

    drive = DrivingField(
        amplitude=Field(time_series=omega, pattern=Pattern("uniform")),
        phase=Field(time_series=phase, pattern=Pattern("uniform")),
        detuning=Field(time_series=detuning, pattern=Pattern("uniform")),
    )

    hamiltonian = Hamiltonian() + drive
    return AnalogHamiltonianSimulation(register=register, hamiltonian=hamiltonian)

In [ ]:
quera_results = []

try:
    n_atoms = 10
    est = quera_estimated_success_from_calibration("quera_aquila", n_atoms)
    target_family = target_family_checkerboard(n_atoms)
    aquila = devices["quera_aquila"]
    program = build_minimal_quera_program(n_atoms)

    # task = aquila.run(program, shots=200)
    # result = task.result()
    # act = empirical_ahs_success(result, target_family)
    # quera_results.append({
    #     "device_key": "quera_aquila",
    #     "provider": "QuEra",
    #     "benchmark": f"ahs_checkerboard_{n_atoms}",
    #     "n_qubits": n_atoms,
    #     "estimated_fidelity": est,
    #     "actual_fidelity": act,
    #     "skew_abs": abs(act - est) if est is not None else None,
    #     "task_id": task.id,
    # })

    print("QuEra estimated_fidelity scaffold:", est)
    print("Target family:", target_family)
except Exception as e:
    print("QuEra benchmark scaffold setup failed:", e)

In [ ]:
frames = []
if len(gate_results) > 0:
    frames.append(pd.DataFrame(gate_results))
if len(quera_results) > 0:
    frames.append(pd.DataFrame(quera_results))

results_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=[
        "device_key", "provider", "benchmark", "n_qubits",
        "estimated_fidelity", "actual_fidelity", "skew_abs", "task_id"
    ]
)

results_df

In [ ]:
if not results_df.empty:
    display(results_df[[
        "provider", "benchmark", "n_qubits",
        "estimated_fidelity", "actual_fidelity", "skew_abs"
    ]].sort_values(["provider", "n_qubits"]))

    print("\nMean absolute skew by provider")
    display(results_df.groupby("provider", as_index=False)["skew_abs"].mean())
else:
    print("No completed runs yet.")

In [ ]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
csv_path = f"fidelity_benchmark_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print("Wrote:", csv_path)